In [ ]:
'''
Brands: Nike (1200), NIKE (800), Adidas (950), adidas (600), Puma (750),Jordan (100)
Group: (Nike, NIKE,Jordan), (Adidas, adidas), (adidas, ADIDAS)
Output: Nike (2100), Adidas (1550)

python and pyspark program
'''


In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
conf = SparkConf().setAppName("Nikeround2").setMaster("local[2]")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/14 01:02:01 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/10/14 01:02:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/14 01:02:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read\
          .text("files/brands")
df.show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------------------+
|               value|
+--------------------+
|Nike (1200), NIKE...|
+--------------------+



### First Reading the brands file and formatting it 

In [4]:
df.select(F.split(F.col("value"),",")).show(truncate=False)

+-----------------------------------------------------------------------------------+
|split(value, ,, -1)                                                                |
+-----------------------------------------------------------------------------------+
|[Nike (1200),  NIKE (800),  Adidas (950),  adidas (600),  Puma (750), Jordan (100)]|
+-----------------------------------------------------------------------------------+



In [6]:
df1 = df.select(
        F.explode(F.split(F.col("value"),",")).alias("col1")
        )
df1.show()

+-------------+
|         col1|
+-------------+
|  Nike (1200)|
|   NIKE (800)|
| Adidas (950)|
| adidas (600)|
|   Puma (750)|
| Jordan (100)|
+-------------+



In [7]:
df_brands = df1.select(
     F.split(F.trim(F.col("col1"))," ")[0].alias("brand"),
     F.regexp_replace(F.split(F.trim(F.col("col1"))," ")[1], r'[()]','').alias("count"),
          )
df_brands.show()

+------+-----+
| brand|count|
+------+-----+
|  Nike| 1200|
|  NIKE|  800|
|Adidas|  950|
|adidas|  600|
|  Puma|  750|
|Jordan|  100|
+------+-----+



### Now Reading the Groups file and Formatting it

In [8]:
df_1 = spark.read\
          .text("files/groups")

df_1.show(truncate=False)

+---------------------------------------------+
|value                                        |
+---------------------------------------------+
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)|
+---------------------------------------------+



In [10]:


df_split = df_1.withColumn("split", F.explode(F.split(F.col("value"),"\),")))

df_split.show(truncate=False)
# df_split = df_1.withColumn("groups", F.split("value", "), "))

# df_split.show(truncate=False)


<>:1: SyntaxWarning: invalid escape sequence '\)'
<>:1: SyntaxWarning: invalid escape sequence '\)'
/tmp/ipykernel_2167/2017096891.py:1: SyntaxWarning: invalid escape sequence '\)'
  df_split = df_1.withColumn("split", F.explode(F.split(F.col("value"),"\),")))


+---------------------------------------------+-------------------------+
|value                                        |split                    |
+---------------------------------------------+-------------------------+
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)|(Nike, NIKE,Jordan       |
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)| (Adidas, adidas, ADIDAS)|
+---------------------------------------------+-------------------------+



In [11]:
df_groups = df_split.withColumn("brand", F.split(F.trim(F.col("split")), ",\s*")[0])\
                   .withColumn("subbrands", F.explode(F.split(F.trim(F.col("split")), ",\s*")))
df_groups.show(truncate=False)



<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2167/509879424.py:1: SyntaxWarning: invalid escape sequence '\s'
  df_groups = df_split.withColumn("brand", F.split(F.trim(F.col("split")), ",\s*")[0])\
/tmp/ipykernel_2167/509879424.py:2: SyntaxWarning: invalid escape sequence '\s'
  .withColumn("subbrands", F.explode(F.split(F.trim(F.col("split")), ",\s*")))


+---------------------------------------------+-------------------------+-------+---------+
|value                                        |split                    |brand  |subbrands|
+---------------------------------------------+-------------------------+-------+---------+
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)|(Nike, NIKE,Jordan       |(Nike  |(Nike    |
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)|(Nike, NIKE,Jordan       |(Nike  |NIKE     |
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)|(Nike, NIKE,Jordan       |(Nike  |Jordan   |
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)| (Adidas, adidas, ADIDAS)|(Adidas|(Adidas  |
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)| (Adidas, adidas, ADIDAS)|(Adidas|adidas   |
|(Nike, NIKE,Jordan), (Adidas, adidas, ADIDAS)| (Adidas, adidas, ADIDAS)|(Adidas|ADIDAS)  |
+---------------------------------------------+-------------------------+-------+---------+



In [12]:
df_group =  df_groups.select(
    F.trim(F.regexp_replace(F.col("brand"), r"[()]", "")).alias("brand"), 
    F.trim(F.regexp_replace(F.col("subbrands"), r"[()]", "")).alias("subbrands"))

df_group.show()

+------+---------+
| brand|subbrands|
+------+---------+
|  Nike|     Nike|
|  Nike|     NIKE|
|  Nike|   Jordan|
|Adidas|   Adidas|
|Adidas|   adidas|
|Adidas|   ADIDAS|
+------+---------+



In [13]:
df_brands.show(truncate = False)

+------+-----+
|brand |count|
+------+-----+
|Nike  |1200 |
|NIKE  |800  |
|Adidas|950  |
|adidas|600  |
|Puma  |750  |
|Jordan|100  |
+------+-----+



## Final Solution

In [19]:
df_final = df_group.alias("t1").join(df_brands.alias("t2"),
                          F.col("t1.subbrands") == F.col("t2.brand"),
                          'left')\
                    .groupBy(F.col("t1.brand"))\
                    .agg(F.sum(F.col("t2.count")).alias("total"))

df_final.show()

+------+------+
| brand| total|
+------+------+
|  Nike|2100.0|
|Adidas|1550.0|
+------+------+



## Melwin Remember in this case we are not supposed to give F.lower in this condition. If we give it becoms cartition product